# domain_title summary statistics of model performance

In [ ]:
chunk_start
import glob

from oceanval.quiet_nc import nc

In [ ]:
%%capture --no-display

show_map = False

if len([x for x in glob.glob("*.ipynb") if "summary_shelf" in x]) > 0:
    ds_regions = nc.open_data(f"{data_dir}/amm7_val_subdomains.nc")
    ds_regions.subset(variables = ["Shelf", "Ocean"])
    ds_regions.sum_all()
    ds_regions.as_missing(0)
    
    ds_plot = ds_regions.pub_plot(legend_position=None, land = "lightgrey")
    land_below()
    show_map = True
try:
    ensemble = nc.create_ensemble("out_dir_value/oceanval_results/annual_mean")
except:
    ensemble = None


In [ ]:
if ensemble is not None:
    if show_map:
        md(f"**Figure {i_figure}**: Map of the area used for the evaluation.")

In [ ]:
if ensemble is not None and wanted("taylor"):
    md("## Taylor diagrams for the sea surface")

In [ ]:
from oceanval.quiet_nc import nc
import pandas as pd
import geopandas as gpd
from IPython.display import display_markdown 
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import os
import glob as glob
from oceanval.utils import restrict_r_to_conda
restrict_r_to_conda()
%load_ext rpy2.ipython

i_table = 1
stamp = nc.session_info["stamp"]
out = ".trackers/" + stamp
if not os.path.exists(".trackers"):
    os.makedirs(".trackers")
# save out as empty file
with open(out, 'w') as f:
    f.write("")

def source_label(title, variable, source, variables):
    # a variable validated against more than one source is labelled with
    # its source, so the comparisons are not merged together
    if variables.count(variable) > 1:
        return f"{title} ({source.upper()})"
    return title

In [ ]:
if ensemble is not None:
    df_taylor = []
    short_names = []
    annual_variables = [os.path.basename(x).split("_")[1] for x in ensemble]
    for ff in ensemble:
        variable = os.path.basename(ff).split("_")[1].replace(".nc", "")
        source = os.path.basename(ff).split("_")[2].replace(".nc", "")
        ds_ff = nc.open_data(ff)
        ff_def = ff.replace(".nc", ".pkl")
        with open(ff_def, "rb") as f:
            ff_definitions = pd.read_pickle(f)
        short_title = source_label(ff_definitions[variable].short_title, variable, source, annual_variables)
        short_names.append(ff_definitions[variable].short_name)

        df_ff = ds_ff.to_dataframe().reset_index()
        lon_name = [df_ff.columns[i] for i in range(len(df_ff.columns)) if "lon" in df_ff.columns[i]][0]
        lat_name = [df_ff.columns[i] for i in range(len(df_ff.columns)) if "lat" in df_ff.columns[i]][0]
        df_taylor.append(
            df_ff
            .loc[:,[lon_name, lat_name, "model", "observation"]]
            .dropna()
            .assign(variable = short_title)
        )
    df_taylor = pd.concat(df_taylor).reset_index(drop=True)

    # fix name of variable

    variables = df_taylor.variable.unique()

    plot_taylor = True
else:
    plot_taylor = False
    df_taylor = None
    df_cor = None
    global_grid = None
# plot_taylor says whether there are gridded matchups, which the sections
# below need; show_taylor whether the user kept the diagram (validate's omit)
show_taylor = plot_taylor and wanted("taylor")

In [ ]:
%%capture --no-display
%%R -i df_taylor -w 600 -h 600 -r 120 -i plot_taylor -i show_taylor

if(show_taylor){
library(plotrix, warn.conflicts = FALSE, quietly = TRUE)
library(dplyr, warn.conflicts = FALSE, quietly = TRUE)

# get unique variable from df_taylor

variables <- df_taylor %>%
    group_by(variable) %>%
    summarize(nsd = sd(model)/sd(observation))  %>%
    arrange(desc(nsd)) %>%
    pull(variable) 

pch = 1:length(variables)
col = rainbow(length(variables))

r_min <- df_taylor %>%
    group_by(variable) %>%
    summarise(r = cor(observation, model, use = "complete.obs")) %>%
    summarize(r = min(r)) %>%
    pull(r)

pos_cor = r_min >= -0.1

i <- 1
for (vv in variables){

    df_vv <- df_taylor %>%
        filter(variable == vv)
    if(i == 1){
        plot_size <- df_vv %>%
        # get the standard deviation of all
        summarize(nsd = sd(model)/sd(observation)) %>%
        pull(nsd)

    }

    if (i == 1){
        taylor.diagram(df_vv$observation, df_vv$model, pch = pch[i], col = col[i], add = FALSE, normalize = TRUE,
        pos.cor = pos_cor, main = NULL

        )
    } else {
        taylor.diagram(df_vv$observation, df_vv$model, pch = pch[i], col = col[i], add = TRUE, normalize = TRUE,
        pos.cor = pos_cor

        )
    }

i <- i + 1
}

legend( plot_size * 1.3, plot_size * 1.9, legend = variables, pch = pch, col = col, bty = "n")




}

In [ ]:
if show_taylor:
    md_basic(f"**Figure {i_figure}**: Taylor diagram for **sea surface** annual mean of {', '.join(variables)}. This diagram compares climatological annual averages of the model and observations across the model's spatial domain. Standard devaiation is normalized by the standard deviation of the observations, and a standard deviation below 1 indicates that the model is less variable than the observations. Note: This figure summarizes the overall ability of the model to reproduce climatological **spatial patterns**, and it does not represent temporal performance.") 
    i_figure += 1

In [ ]:
if plot_taylor and wanted("bias_table"):
    md("## Model biases based on gridded sea surface data")

In [ ]:
if plot_taylor and wanted("bias_table"):
    from oceanval import skill
    df_bias= []
    for ff in ensemble:
        variable = os.path.basename(ff).split("_")[1].replace(".nc", "")
        source = os.path.basename(ff).split("_")[2].replace(".nc", "")
        key = variable
        ds_ff = nc.open_data(ff)
        ds_ff.set_precision("F32")
        ff_def = ff.replace(".nc", ".pkl")
        with open(ff_def, "rb") as f:
            ff_definitions = pd.read_pickle(f)
        variable = source_label(ff_definitions[variable].short_title, variable, source, annual_variables)

        ds_ff.assign(bias = lambda x: x.model - x.observation)
        ds_ff.spatial_mean()
        bias = ds_ff.to_dataframe().reset_index().bias.values[0]
        unit = ds_ff.contents.unit[0]
        name = variable 
        model = ds_ff.to_dataframe().reset_index().model.values[0]
        observation = ds_ff.to_dataframe().reset_index().observation.values[0]
        df_bias.append(pd.DataFrame({"Variable": [name], "Modelled spatial mean": [model], "Observational spatial mean":[observation], "Model bias": [bias], "Unit": [unit]}))
        # chlorophyll and kd490 are roughly log-normal, so their means are also given on log10 values
        if skill.is_log(key):
            s_log = skill.file_stats(ff, log = True)
            df_bias.append(pd.DataFrame({"Variable": [f"{name} (log10)"], "Modelled spatial mean": [s_log["model_mean"]], "Observational spatial mean":[s_log["obs_mean"]], "Model bias": [s_log["bias"]], "Unit": ["log10"]}))
    df_bias = pd.concat(df_bias).reset_index(drop=True)
    df_bias = df_bias.assign(percentage_bias = lambda x: x["Model bias"]/x["Observational spatial mean"]*100)
    df_bias.loc[df_bias.Variable == "Temperature", "percentage_bias"] = np.nan
    # a percentage of a log has no meaning
    df_bias.loc[df_bias.Unit == "log10", "percentage_bias"] = np.nan
    log_rows = (df_bias.Unit == "log10").any()
    df_bias.columns = ["Variable", "Model mean", "Observed mean", "Model bias", "Unit", "Percentage bias"]
    # tidy Variable
    # Make everything 2 dp, except for 1st column
    df_bias.iloc[:,1:] = df_bias.iloc[:,1:].round(2)
    # drop unit
    df_bias = df_bias.drop(columns=["Unit"])
    df_display(df_bias)

In [ ]:
if plot_taylor and wanted("bias_table"):
    log_text = " Rows marked log10 are worked out on the log10 of the values (cells where either is not positive are left out), as chlorophyll and kd490 are roughly log-normal; their percentage bias is not given." if log_rows else ""
    md(f"**Table {i_table}**: Bias of model compared with **sea surface** observations. The bias is calculated as the modelled spatial mean minus the observational spatial mean. The percentage bias is calculated as the model bias divided by the observational spatial mean.{log_text}")
    i_table += 1

In [ ]:
if plot_taylor and wanted("spatial_table"):
    md("## Spatial performance of the model at the sea surface") 

In [ ]:
if plot_taylor and wanted("spatial_table"):
    from oceanval import skill
    df_cor = []
    short_names = []
    for ff in ensemble:
        variable = os.path.basename(ff).split("_")[1].replace(".nc", "")
        source = os.path.basename(ff).split("_")[2].replace(".nc", "")
        key = variable
        ff_def = ff.replace(".nc", ".pkl")
        with open(ff_def, "rb") as f:
            ff_definitions = pd.read_pickle(f)
        short_names.append(ff_definitions[variable].short_name)
        variable = source_label(ff_definitions[variable].short_title, variable, source, annual_variables)

        if variable.lower() == "sst":
            variable = "SST"
        # area-weighted, so that r, the standard deviations and the centred RMSD
        # are those of a Taylor diagram: cRMSD² = σo² + σm² − 2σoσm·r
        rows = [(variable, skill.file_stats(ff))]
        if skill.is_log(key):
            rows.append((f"{variable} (log10)", skill.file_stats(ff, log = True)))
        for label, s in rows:
            df_cor.append(pd.DataFrame({
                "Variable": [label],
                "Correlation": [s["r"]],
                "RMSD": [s["rmsd"]],
                "Normalised standard deviation": [s["nsd"]],
                "Centred RMSD": [s["crmsd"]],
                "Normalised centred RMSD": [s["crmsd_n"]],
            }))
    df_cor = pd.concat(df_cor).reset_index(drop=True)
    log_rows = df_cor.Variable.str.endswith("(log10)").any()
    df_cor.columns = [
        "Variable",
        "Spatial correlation between model and observations",
        "Spatial RMSD between model and observations",
        "Normalised standard deviation (σm/σo)",
        "Centred RMSD",
        "Centred RMSD/σo",
    ]
    df_cor.iloc[:, 1:] = df_cor.iloc[:, 1:].astype(float).round(3)

    # tidy Variable
    df_display(df_cor)

In [ ]:
if plot_taylor and wanted("spatial_table"):
    log_text = " Rows marked log10 are worked out on the log10 of the values (cells where either is not positive are left out)." if log_rows else ""
    md(f"**Table {i_table}**: Pearson correlation coefficient, RMSD, normalised standard deviation and centred RMSD between model and observations at the **sea surface** for annual mean of {', '.join(short_names)}. This table compares climatological annual averages of the model and observations across the model's spatial domain. Every statistic is weighted by the area of the grid cells. The normalised standard deviation is the model's standard deviation divided by the observations' (below 1, the model varies less in space than the observations). The centred RMSD is the RMSD once each mean is removed, so it is the part of the RMSD that the bias does not explain; with the correlation and the normalised standard deviation, it gives a variable's position on a Taylor diagram as numbers.{log_text}")
    i_table += 1

In [ ]:
if plot_taylor and (wanted("temporal_maps") or wanted("temporal_table")):
    md("## Temporal performance of the model at the sea surface")

In [ ]:
if plot_taylor:
    global_grid = False
    paths =   glob.glob("out_dir_value/oceanval_results/temporals/*.nc")
    for ff in paths:
        ds_ff = nc.open_data(ff)
        df_ff = ds_ff.to_dataframe().reset_index()
        lat_name = [df_ff.columns[i] for i in range(len(df_ff.columns)) if "lat" in df_ff.columns[i]][0]
        lat_min = df_ff[lat_name].values.min()
        lat_max = df_ff[lat_name].values.max()
        if lat_min < -89 and lat_max > 89:
            global_grid = True


if plot_taylor and wanted("temporal_maps"):
    # correlation maps for panel_plot, titled by variable
    cor_panels = dict()
    paths =  glob.glob("out_dir_value/oceanval_results/temporals/*.nc")
    temporal_variables = [os.path.basename(x).split("_")[0] for x in paths]
    for ff in paths:
        ds_ff = nc.open_data(ff)
        if global_grid:
            ds_ff.to_latlon(lon = [-179.5, 179.5], lat = [-89.5, 89.5], res = 1)
        variable = os.path.basename(ff).split("_")[0].replace(".nc", "")
        source = os.path.basename(ff).split("_")[2].replace(".nc", "")
        variable = source_label(variable.title(), variable, source, temporal_variables)
        cor_panels[variable] = ds_ff

In [ ]:
if plot_taylor and wanted("temporal_maps"):
    # in alphabetical order
    panels = {
        re.sub("CO2|co2|CO_2", "CO$_2$", key): cor_panels[key]
        for key in sorted(cor_panels, key = str.lower)
    }
    nc.panel_plot(
        panels,
        var = "cor",
        ncol = 3 if len(panels) == 3 else None,
        legend = "Correlation coefficient",
        legend_position = "bottom",
        land = "lightgrey",
        grid_labels = False,
    )
    land_below()

In [ ]:
if plot_taylor and wanted("temporal_maps"):
    md(f"**Figure {i_figure}**: Temporal correlation (Pearson correlation coefficient) between model and observations for {', '.join(short_names)}. The correlation is calculated in each grid cell between the climatological monthly averages of the model and observations, so it shows how well the model reproduces the seasonal cycle across the model's spatial domain.")
    i_figure += 1

In [ ]:
if plot_taylor and wanted("temporal_table"):
    md("The overall ability of the model reproduce the seasonality of each variable was estimated by calculating the spatial mean of the Pearson correlation coefficient between the model and the observations. The spatial mean was calculated by averaging the correlation coefficient of each grid cell.")

In [ ]:
if plot_taylor and wanted("temporal_table"):
    df_cor = []
    paths = glob.glob("out_dir_value/oceanval_results/temporals/*.nc")
    short_names = []
    temporal_variables = [os.path.basename(x).split("_")[0] for x in paths]
    for ff in paths:
        ds_ff = nc.open_data(ff, checks = False)
        ds_ff.spatial_mean()
        variable = os.path.basename(ff).split("_")[0].replace(".nc", "")
        source = os.path.basename(ff).split("_")[2].replace(".nc", "")
        ff_def = ff.replace(".nc", ".pkl")
        with open(ff_def, "rb") as f:
            ff_definitions = pd.read_pickle(f)
        short_names.append(ff_definitions[variable].short_name)
        variable = source_label(ff_definitions[variable].short_title, variable, source, temporal_variables)

        df_cor.append(pd.DataFrame({"Variable": [variable], "Correlation": [ds_ff.to_dataframe().reset_index().cor.values[0]]}))
    
    df_cor = pd.concat(df_cor).reset_index(drop=True)
    
    
    
    
    # tidy Variable



In [ ]:
# Make everything 2 dp, except for 1st column
if plot_taylor and wanted("temporal_table"):
    df_display(df_cor)

In [ ]:
if plot_taylor and wanted("temporal_table"):
    md(f"**Table {i_table}**: Spatial average of the temporal correlation (Pearson correlation coefficient) between model and observations for annual mean of {', '.join(short_names)}. The correlation is calculated for each grid cell individually using monthly climatological averages. The spatial average is then calculated for each variable.")
    i_table += 1

In [ ]:
# the gridded matchups' totals over the cells matched up (the gap report's option 34), as each
# gridded page wrote them (oceanval.totals), with the published ranges of the global totals
totals_paths = sorted(glob.glob("out_dir_value/oceanval_results/totals/*_totals.csv"))
if len(totals_paths) > 0 and wanted("global_totals"):
    from oceanval import totals
    df_totals = pd.concat([pd.read_csv(x) for x in totals_paths]).reset_index(drop = True)
    rows = []
    any_outside = False
    for _, row in df_totals.iterrows():
        refs = totals.references(row.variable)
        ranges = "; ".join(f"{x['low']:g}–{x['high']:g} ({x['citation']}{', ' + x['period'] if x['period'] else ''})" if x["low"] != x["high"] else f"about {x['low']:g} ({x['citation']})" for x in refs) or "None found"
        outside = totals.outside(row.model, [x for x in refs if x["low"] != x["high"]])
        any_outside = any_outside or outside
        cover = f"{row.area_km2 / 1e6:.1f} million km<sup>2</sup>" if row.kind == totals.FLUX else f"{row.volume_km3 / 1e6:.1f} million km<sup>3</sup>"
        rows.append({
            "Variable": f"{row.variable} ({str(row.source).upper()})",
            "Unit": row.units,
            "Model": f"{row.model:.3g}" + (" *" if outside else ""),
            "Observations": f"{row.observation:.3g}",
            "Area or volume matched": cover,
            "Published global total": ranges,
        })
    md("## Totals over the area matched up")
    md("The table below gives each gridded matchup of a flux (summed over the area, as a rate for the year) or of a concentration through the water column (summed over the volume, as an inventory) over the grid cells with both model and observational data, beside the published estimates of the global total. A matchup covers only the cells with observations, so its total is lower than a global one unless they cover the whole ocean: the area or volume they cover is given.")
    df_display(pd.DataFrame(rows))
    md(f"**Table {i_table}**: Totals of the model and the observations over the grid cells matched up." + (" A model total marked * is outside every published range." if any_outside else "") + " Each published range is quoted from the paper cited.")
    i_table += 1


In [ ]:

# the depth summaries of the point matchups, and of the gridded ones through the water column
depth_paths = glob.glob("out_dir_value/oceanval_results/*_depth_summary.csv")
gridded_paths = sorted(x for x in depth_paths if x.endswith("_gridded_depth_summary.csv"))
paths = sorted(x for x in depth_paths if not x.endswith("_gridded_depth_summary.csv"))

In [ ]:
if len(paths) + len(gridded_paths) > 0 and wanted("depth_performance"):
    md("## Perfomance of model across depths")
    md("Root mean square deviation (RMSD), bias and correlation between model and observations were calculated for each variable at different depths. The RMSD is calculated as the square root of the mean of the squared differences between the model and observations. The bias is calculated as the modelled spatial mean minus the observational spatial mean. The correlation is calculated for each variable at different depths. A gridded matchup's depth levels are put in the same depth ranges as the point observations, each level weighted by the area of its grid cells and the thickness of the layer it stands for.") 
df_map = []
if len(paths) > 0 and wanted("depth_performance"):
    # the map leaves out the deepest bin if it has no max (see oceanval.depths)
    map_bins = [pair for pair in depth_bins if pair[1] is not None]
    depth_variables = [os.path.basename(x).split("_")[0] for x in paths]
    for ff in paths:
        vv = os.path.basename(ff).split("_")[0]
        source = os.path.basename(ff).split("_")[1]
        ff_points = glob.glob(f"data_dir_value/oceanval_matchups/point/all/{vv}/{source}/*all*_{vv}.csv")
        if len(ff_points) == 0:
            continue
        ff_points = ff_points[0]
        ff_def = ff_points.replace(".csv", "_definitions.pkl")
        with open(ff_def, "rb") as f:
            definitions = pd.read_pickle(f)
        vv = source_label(definitions[vv].short_title, vv, source, depth_variables)
        df_ff = pd.read_csv(ff_points).loc[:,["lon", "lat", "depth"]]
        lon_min = lon_lim[0]
        lon_max = lon_lim[1]
        lat_min = lat_lim[0]
        lat_max = lat_lim[1]
        # filter by lon and lat
        df_ff = df_ff.query(f"lon >= {lon_min} and lon <= {lon_max} and lat >= {lat_min} and lat <= {lat_max}").reset_index(drop = True)
        df_ff = df_ff.assign(depth_bin = df_ff["depth"].apply(bin_depth, args = (map_bins,)))
        # add variable
        df_ff = df_ff.assign(variable = vv)
        df_map.append(df_ff)
if len(df_map) > 0:
    df_map = pd.concat(df_map).reset_index(drop=True)
    # drop na
    df_map = df_map.dropna()
    plot_map = True
else:
    df_map = None
    plot_map = False

In [ ]:
%%capture --no-display
%%R -i df_map -i plot_map -i depth_levels -w 1000 -h 700
if(plot_map){
library(ggplot2, warn.conflicts = FALSE, quietly = TRUE)
    library(tidyverse, warn.conflicts = FALSE, quietly = TRUE)
# a small area, a lake for example, needs a map with more detail than the world map has
if (diff(range(df_map$lon, na.rm = TRUE)) > 50 | diff(range(df_map$lat, na.rm = TRUE)) > 50 | !requireNamespace("mapdata", quietly = TRUE)) {
    world_map <- map_data("world")
} else {
    library(mapdata, warn.conflicts = FALSE, quietly = TRUE)
    world_map <- tryCatch(
        map_data("worldHires", xlim = range(df_map$lon, na.rm = TRUE) + c(-1, 1), ylim = range(df_map$lat, na.rm = TRUE) + c(-1, 1)) %>% filter(region != "Great Lakes"),
        error = function(e) map_data("world")
    )
}

xlim <- c(min(df_map$lon), max(df_map$lon))
ylim <- c(min(df_map$lat), max(df_map$lat))
df_map <- df_map %>%
    mutate(depth_bin = factor(depth_bin, levels = unlist(depth_levels)))


gg <- ggplot(df_map)+
        geom_point(aes(x  = lon,y =   lat), size = 0.5)+ 
        coord_cartesian(xlim = xlim, ylim = ylim, expand = FALSE)+
        theme_bw(base_size = 24)+
        facet_grid(variable~depth_bin)+
        labs(color = "Depth bin")+
        theme_bw(base_family = "Helvetica", base_size = 8) +
        theme(
          legend.position = "bottom", legend.direction = "horizontal", legend.box = "horizontal", legend.key.width = unit(3.0, "cm"),
          legend.key.height = unit(0.5, "cm")
        ) +
        labs(x = NULL, y = NULL) +
        theme(plot.margin = unit(c(2, 0, 2, 0), "mm")) +
        theme(plot.title = element_text(hjust = 0.5))
        # make the legend 3 cm wide
        # theme( legend_key_size = unit(3, "cm"))

y_labels <-  as.numeric(na.omit(layer_scales(gg)$y$break_positions()))
x_labels <- as.numeric(na.omit(layer_scales(gg)$x$break_positions()))
x_breaks <- x_labels
y_breaks <- y_labels

# y labels are north-south coordinates. Make them more appropriate
# i.e. 10 should be 10 °N, -10 should be 10 °S

y_labels <- ifelse(y_labels >= 0, paste0(y_labels, "°N"), paste0(abs(y_labels), "°S"))
x_labels <- ifelse(x_labels >= 0, paste0(x_labels, "°E"), paste0(abs(x_labels), "°W"))

gg <- gg + scale_y_continuous(breaks = y_breaks, labels = y_labels) +
          scale_x_continuous(breaks = x_breaks, labels = x_labels)

# the land is drawn first, so the points are above it. It is added now, as the
# breaks above are those of the points
gg$layers <- c(
    list(geom_polygon(data = world_map, aes(x = long, y = lat, group = group), fill = "grey", colour = "grey")),
    gg$layers
)
gg
}


In [ ]:
if plot_map:
    md(f"**Figure {i_figure}**: Map of the locations of matchups at each depth range.") 
    i_figure += 1

In [ ]:


def depth_tables(statistic):
    """The point and gridded depth summaries of one statistic, a row for each
    matchup and a column for each depth range."""
    point_variables = [os.path.basename(x).split("_")[0] for x in paths]
    rows = []
    for ff in paths + gridded_paths:
        df = pd.read_csv(ff)
        vv_ff = os.path.basename(ff).split("_")[0]
        source_ff = os.path.basename(ff).split("_")[1]
        if ff in gridded_paths:
            vv_ff = f"{vv_ff} ({source_ff.upper()}, gridded)"
        else:
            vv_ff = source_label(vv_ff, vv_ff, source_ff, point_variables)
        if "unit" not in df.columns:
            df = df.assign(unit = "")
        df = df.assign(unit = df.unit.fillna(""))
        rows.append(
            df
            .loc[:,["Depth", statistic, "unit"]]
            .pivot(index = "unit", columns = "Depth", values = statistic)
            .assign(variable = vv_ff)
            .reset_index()
        )
    df_out = pd.concat(rows).reset_index(drop=True)
    if statistic == "Correlation":
        df_out = df_out.drop(columns = ["unit"])
    first = ["variable"] if statistic == "Correlation" else ["variable", "unit"]
    locs = [x for x in dict.fromkeys([*first, *depth_levels, "0-150m"]) if x in df_out.columns]
    return df_out.loc[:,locs]


if len(paths) + len(gridded_paths) > 0 and wanted("depth_performance"):

    df_depth = depth_tables("RMSD")
    df_display(df_depth)
    md(f"**Table {i_table}**: Root mean square deviation (RMSD) of model compared with observations at different depths. The RMSD is calculated as the square root of the mean of the squared differences between the model and observations.")
    i_table += 1

    # now do bias
    df_bias = depth_tables("Bias")
    df_display(df_bias)
    md(f"**Table {i_table}**: Bias of model compared with observations at different depths.")

    i_table += 1

    # now do correlation
    df_cor = depth_tables("Correlation")
    df_display(df_cor)
    md(f"**Table {i_table}**: Pearson correlation coefficient between model and observations at different depths. The correlation is calculated for each variable at different depths.")
    i_table += 1